In [2]:
import re, random, warnings
import numpy as np
import pandas as pd
import nltk
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

TAXONOMY = {
    "Academics": {
        "course_information": "Course registration, prerequisites, credits, and advising",
        "exam_information": "Exam schedules, assessment rules, and results",
    },
    "Campus Services": {
        "library_services": "Library hours, borrowing, renewals, and study spaces",
        "it_support": "Wi-Fi, accounts, passwords, and learning platform access",
    },
    "Administration": {
        "fees_financial_aid": "Tuition payments, invoices, scholarships, and aid",
        "documents_records": "Transcripts, enrollment letters, and student records",
    },
    "Wellbeing": {
        "emergency_help": "Immediate danger, urgent medical or safety assistance",
        "wellbeing_support": "Counselling, accessibility, and non-urgent wellbeing support",
    },
}

# Each list has eight distinct example queries for its intent.
EXAMPLES = {
"course_information": [
"How do I register for an elective course?", "What are the prerequisites for data science?", "Can an advisor help me choose my courses?", "How many credits is this semester?", "Where can I see the course catalog?", "When does course registration open?", "Can I change my major requirements?", "How do I add a class to my timetable?"],
"exam_information": [
"When is the midterm exam scheduled?", "Where can I find the final examination timetable?", "How are exam results published?", "What is the policy for a missed exam?", "Can I request exam accommodations?", "When will grades be released?", "Where is my examination room listed?", "How do I apply for a recheck of my paper?"],
"library_services": [
"What time does the library close tonight?", "How can I renew a borrowed book?", "Where can I reserve a study room?", "How many books may I borrow?", "Can I access journals online?", "Is the library open on Sunday?", "How do I return an overdue book?", "Where are quiet study spaces?"],
"it_support": [
"I cannot connect to campus Wi-Fi", "How do I reset my student password?", "My learning portal login is not working", "Where can I get help with my university account?", "How do I install campus software?", "My student email is unavailable", "I forgot my network username", "How can I report a computer problem?"],
"fees_financial_aid": [
"How can I pay my tuition fees?", "Where do I download my fee invoice?", "When is the tuition payment deadline?", "How do I apply for a scholarship?", "Can I set up a payment plan?", "Why has my financial aid not arrived?", "How can I check my account balance?", "Are late payment charges applied?"],
"documents_records": [
"How do I request an official transcript?", "Can I get a proof of enrollment letter?", "Where can I update my personal details?", "How do I obtain a student ID card?", "Can the registrar send my academic record?", "How long does a certificate request take?", "Where do I correct my name on records?", "How can I get a graduation letter?"],
"emergency_help": [
"There is a fire in the residence hall", "Someone is badly injured and needs an ambulance", "I am in immediate danger on campus", "Please call campus security for an emergency", "There is a serious medical emergency", "I see smoke coming from a campus building", "Someone is threatening me right now", "I need urgent help because a person collapsed"],
"wellbeing_support": [
"How can I book a counselling appointment?", "I feel stressed and want to talk to someone", "Where can I find mental health support?", "How do I contact accessibility services?", "Can I speak with a student wellbeing advisor?", "I need support managing anxiety", "Where can I ask about disability accommodations?", "Are there peer support groups for students?"],
}
rows = [{"query": q, "intent": label} for label, queries in EXAMPLES.items() for q in queries]
corpus = pd.DataFrame(rows)
assert len(corpus) >= 40 and corpus.intent.nunique() >= 5
print(f"Corpus: {len(corpus)} queries; {corpus.intent.nunique()} leaf intents")
display(corpus.groupby("intent").size().rename("queries").to_frame())
display(corpus.sample(8, random_state=42))


Corpus: 64 queries; 8 leaf intents


,queries
intent,
course_information,8
documents_records,8
emergency_help,8
exam_information,8
fees_financial_aid,8
it_support,8
library_services,8
wellbeing_support,8


,query,intent
52,There is a serious medical emergency,emergency_help
58,Where can I find mental health support?,wellbeing_support
0,How do I register for an elective course?,course_information
44,Can the registrar send my academic record?,documents_records
5,When does course registration open?,course_information
36,Can I set up a payment plan?,fees_financial_aid
16,What time does the library close tonight?,library_services
12,Can I request exam accommodations?,exam_information


In [3]:
CONCEPTS = {
    "Student": "A person enrolled at the university",
    "Course": "A taught unit that students can register for",
    "Exam": "An assessment scheduled by an academic department",
    "Library": "The university borrowing and study service",
    "IT Helpdesk": "The team that supports accounts and campus technology",
    "Tuition Account": "A student's fees, invoices, and payment record",
    "Registrar": "The office maintaining enrollment and academic records",
    "Counselling Service": "A service for confidential wellbeing support",
    "Campus Security": "The team responding to urgent campus safety incidents",
    "Student ID": "An identifier used to access university services",
}
RELATIONSHIPS = [
    ("Student", "registers for", "Course"),
    ("Course", "has", "Exam"),
    ("Student", "takes", "Exam"),
    ("Student", "borrows from", "Library"),
    ("Library", "provides access to", "Student"),
    ("Student", "contacts for technical help", "IT Helpdesk"),
    ("Student", "pays", "Tuition Account"),
    ("Tuition Account", "tracks fees for", "Student"),
    ("Registrar", "maintains records for", "Student"),
    ("Student", "requests records from", "Registrar"),
    ("Student", "may contact", "Counselling Service"),
    ("Campus Security", "responds to emergencies involving", "Student"),
    ("Student", "uses", "Student ID"),
    ("Student ID", "grants access to", "Library"),
]
print(f"Concepts: {len(CONCEPTS)} | Relationships: {len(RELATIONSHIPS)}")
for subject, relation, object_ in RELATIONSHIPS:
    print(f"{subject} --{relation}--> {object_}")


Concepts: 10 | Relationships: 14
Student --registers for--> Course
Course --has--> Exam
Student --takes--> Exam
Student --borrows from--> Library
Library --provides access to--> Student
Student --contacts for technical help--> IT Helpdesk
Student --pays--> Tuition Account
Tuition Account --tracks fees for--> Student
Registrar --maintains records for--> Student
Student --requests records from--> Registrar
Student --may contact--> Counselling Service
Campus Security --responds to emergencies involving--> Student
Student --uses--> Student ID
Student ID --grants access to--> Library


In [4]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt

STOP_WORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()
def preprocess(text):
    text = text.lower()                              # lowercase conversion
    tokens = re.findall(r"[a-z]+(?:'[a-z]+)?", text) # tokenization
    tokens = [t for t in tokens if t not in STOP_WORDS] # stopword removal
    return " ".join(LEMMATIZER.lemmatize(t) for t in tokens) # lemmatization

print("Raw:       ", "I cannot connect to campus Wi-Fi")
print("Processed:", preprocess("I cannot connect to campus Wi-Fi"))

X_train, X_test, y_train, y_test = train_test_split(
    corpus.query, corpus.intent, test_size=0.25, random_state=42, stratify=corpus.intent
)
model = Pipeline([
    ("tfidf", TfidfVectorizer(preprocessor=preprocess, token_pattern=r"(?u)\b\w+\b", ngram_range=(1,2))),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
labels = sorted(corpus.intent.unique())
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred, average='macro', zero_division=0):.3f} (macro)")
print(f"Recall:    {recall_score(y_test, y_pred, average='macro', zero_division=0):.3f} (macro)")
print(f"F1-score:  {f1_score(y_test, y_pred, average='macro', zero_division=0):.3f} (macro)")
print("\nPer-class report:\n", classification_report(y_test, y_pred, labels=labels, zero_division=0))
cm = confusion_matrix(y_test, y_pred, labels=labels)
ConfusionMatrixDisplay(cm, display_labels=labels).plot(xticks_rotation=75, cmap="Blues", values_format="d")
plt.title("Intent classifier confusion matrix")
plt.tight_layout()
plt.show()


Raw:        I cannot connect to campus Wi-Fi
Processed: cannot connect campus wi fi


TypeError: Input should have at least 1 dimension i.e. satisfy `len(x.shape) > 0`, got scalar `array(<bound method DataFrame.query of                                                query              intent
0          How do I register for an elective course?  course_information
1       What are the prerequisites for data science?  course_information
2          Can an advisor help me choose my courses?  course_information
3                 How many credits is this semester?  course_information
4                Where can I see the course catalog?  course_information
..                                               ...                 ...
59          How do I contact accessibility services?   wellbeing_support
60     Can I speak with a student wellbeing advisor?   wellbeing_support
61                   I need support managing anxiety   wellbeing_support
62  Where can I ask about disability accommodations?   wellbeing_support
63       Are there peer support groups for students?   wellbein

In [ ]:
KNOWLEDGE_BASE = {
    "course_information": "For course selection and registration, consult the course catalog and your academic advisor. Check prerequisites and registration dates in the student portal.",
    "exam_information": "Exam dates, rooms, assessment rules, and results are published by the academic department or student portal. Contact your instructor about a missed exam or recheck.",
    "library_services": "The library portal lists current opening hours, borrowing limits, renewals, room reservations, and digital journal access. Contact the library desk for account-specific help.",
    "it_support": "For campus Wi-Fi, account, email, or learning portal problems, try the official password reset page and contact the IT helpdesk if access remains unavailable.",
    "fees_financial_aid": "Use the student finance portal to review invoices, balances, deadlines, payment options, and scholarship information. Contact the finance office about account-specific questions.",
    "documents_records": "Request transcripts, enrollment letters, and record corrections through the registrar. Check the registrar's portal for forms, identity requirements, and processing times.",
    "emergency_help": "If anyone is in immediate danger or needs urgent medical help, call your local emergency number now. On campus, contact campus security using the posted emergency number. Move to a safe place if possible.",
    "wellbeing_support": "You can contact the university counselling or student wellbeing service to ask about confidential appointments, accessibility support, and peer programs. If you may harm yourself or someone else, contact emergency services now.",
}

EMERGENCY_TERMS = ["fire", "badly injured", "immediate danger", "emergency", "ambulance", "collapsed", "threatening me", "urgent help", "smoke"]
CONFIDENCE_THRESHOLD = 0.30

def answer_query(query):
    text = query.strip()
    if not text:
        return {"intent": "unsupported_query", "confidence": 0.0, "response": "Please enter a question so I can help."}

    # Rule 1: Emergency escalation precedes model prediction.
    if any(term in text.lower() for term in EMERGENCY_TERMS):
        return {"intent": "emergency_help", "confidence": 1.0,
                "response": KNOWLEDGE_BASE["emergency_help"] + " For non-urgent support, tell me what service you need."}

    # Rules 2 and 3: classify, then handle low confidence and unsupported queries.
    probabilities = model.predict_proba([text])[0]
    best_index = int(np.argmax(probabilities))
    intent = model.classes_[best_index]
    confidence = float(probabilities[best_index])
    if confidence < CONFIDENCE_THRESHOLD:
        return {"intent": "unsupported_query", "confidence": confidence,
                "response": "I’m not confident I understood. I can help with courses, exams, library, IT, fees, student records, or wellbeing. Please rephrase or contact the relevant university office."}

    response = KNOWLEDGE_BASE[intent]
    # Rule 4: direct urgent-risk language to emergency services regardless of intent.
    if re.search(r"\b(suicid|kill myself|hurt myself|harm someone)\b", text.lower()):
        response += " If there is immediate risk, call your local emergency number now."
    # Rule 5: add a human handoff for account-specific administration questions.
    if intent in {"fees_financial_aid", "documents_records", "it_support"}:
        response += " For account-specific details, contact the responsible university office or helpdesk."
    return {"intent": intent, "confidence": confidence, "response": response}

# Rule examples (emergency, confidence fallback, empty input, handoff, wellbeing risk)
for q in ["There is a fire in my building", "What is the capital of France?", "", "How do I pay tuition?", "I want to hurt myself"]:
    result = answer_query(q)
    print(f"\nQ: {q or '[empty]'}\nIntent: {result['intent']} ({result['confidence']:.2f})\nA: {result['response']}")


In [ ]:
import gradio as gr

def respond_to_many(text):
    queries = [line.strip() for line in text.splitlines() if line.strip()]
    if not queries:
        return "Enter one or more questions, each on its own line."
    results = []
    for i, query in enumerate(queries, start=1):
        result = answer_query(query)
        results.append(
            f"### {i}. {query}\n**Intent:** `{result['intent']}`  |  **Confidence:** {result['confidence']:.2f}\n\n{result['response']}"
        )
    return "\n\n---\n\n".join(results)

with gr.Blocks(title="Campus Student Support Assistant") as demo:
    gr.Markdown("# Campus Student Support Assistant\nAsk one question per line. This educational demo gives general guidance; check official university channels for current policies.")
    questions = gr.Textbox(label="Your questions", lines=6, placeholder="How do I renew a library book?\nWhen is course registration open?")
    submit = gr.Button("Get answers", variant="primary")
    output = gr.Markdown(label="Assistant responses")
    submit.click(fn=respond_to_many, inputs=questions, outputs=output)
    questions.submit(fn=respond_to_many, inputs=questions, outputs=output)

demo.launch(share=True, debug=False)
